# 13 — Agents
### LangChain Agents & Tools · Part 3 of 3 (`11_Tools` → `12_Tool_Calling` → `13_Agents`)

In `12_Tool_Calling` you wrote the loop yourself: call the model, run its tool requests, feed the
results back, repeat — with a step limit, retries, error handling and an approval gate. An **agent** is
that same loop, packaged. This notebook uses `create_agent`, shows it is *exactly* your loop (the graph
it builds has just two nodes, `model` and `tools`, wired in a cycle), and — more usefully — shows which
of your protections it gives you for free, and which it quietly **doesn't**.

```
 __start__ ──► model ──── no tool calls ────► __end__
                 ▲  │
                 │  └── tool calls ──► tools ─┐
                 └──────────────────────────────┘
```

> 📌 **Checked while building this notebook** (`langchain 1.4.4`, `langgraph 1.2.14`; each by running it)
> - **The default step limit is effectively none.** LangGraph's default recursion limit is **10007**
>   (read from its source in `langgraph 1.2.14`; it can be changed with the
>   `LANGGRAPH_DEFAULT_RECURSION_LIMIT` environment variable). An agent with no explicit limit can run for
>   thousands of steps — and you pay for each model call.
> - **Mistakes the *model* makes are handled for you; exceptions raised *inside your tool* are not.**
>   A missing argument or an unknown tool name becomes an error message the model can read. But a
>   `ToolException` from your own tool code **crashes the whole run** unless you opt in. Your `12` loop
>   converted both, so the agent is *less* forgiving out of the box.
> - **An approval pause really pauses.** While waiting for a human, the tool had not run. If the
>   reviewer *edits* the call, the **reviewer's** arguments are what execute.
> - **Old tutorials use APIs that have moved.** `create_react_agent` still imports but is deprecated in
>   favour of `create_agent`; `AgentExecutor` can no longer be imported from `langchain.agents` and
>   lives only in `langchain_classic.agents`.
> - **Not run live:** the sandbox this was built in couldn't reach Groq. The cells that call a real
>   model were exercised against a stand-in; everything in Sections 4–8 uses a scripted model built
>   *inside* this notebook and demonstrates real `create_agent` behaviour.
>
> **How this fits with your `LangGraph/` folder:** `create_agent` *builds a LangGraph graph*. Your
> folders `07_Middleware`, `08_HITL`, `09_Memory_Persistence` and `13_Streaming` go deeper on the
> pieces introduced here. This notebook is the bridge between the loop you wrote and those.

## Setup

In [ ]:
# %pip install -U langchain langgraph langchain-groq python-dotenv


In [1]:
import os, json
from datetime import date, timedelta
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain.agents.middleware import (
    ModelCallLimitMiddleware, ToolCallLimitMiddleware, ModelRetryMiddleware,
    ToolErrorMiddleware, HumanInTheLoopMiddleware,
)
from langchain_core.tools import tool, ToolException
from langchain_core.messages import AIMessage
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

print("Ready. Needs GROQ_API_KEY in your .env for the live cells.")


Ready. Needs GROQ_API_KEY in your .env for the live cells.


---
## 1. The same support toolbox as `12`

Copied from the `12_Tool_Calling` capstone so this notebook stands alone. Notice what has not changed:
the refund rules live in **one function** (`refund_check`) and `issue_refund` re-checks them itself. An
agent changes who *drives* the tools, not who is responsible for the rules.

In [2]:
TODAY = date.today()

ORDERS = {
    "A-1001": {"item": "Wireless earbuds",  "status": "delivered", "delivered_on": TODAY - timedelta(days=10), "digital": False},
    "A-1002": {"item": "Running shoes",     "status": "delivered", "delivered_on": TODAY - timedelta(days=45), "digital": False},
    "A-1003": {"item": "E-book bundle",     "status": "delivered", "delivered_on": TODAY - timedelta(days=2),  "digital": True},
    "A-1004": {"item": "Phone case",        "status": "shipped",   "delivered_on": None,                       "digital": False},
    "A-1005": {"item": "Laptop sleeve",     "status": "delivered", "delivered_on": TODAY - timedelta(days=5),  "digital": False},
}
REFUNDED: list[str] = []
TICKETS: list[dict] = []


def get_order(order_id: str) -> dict:
    order = ORDERS.get(order_id.strip().upper())
    if order is None:
        raise ToolException(f"No order found with ID '{order_id}'. Ask the customer to double-check the order number.")
    return order


def refund_check(order_id: str) -> tuple[bool, str]:
    """The ONE place the refund rules live."""
    o = get_order(order_id)
    if o["status"] != "delivered":
        return False, "the order has not been delivered yet"
    if o["digital"]:
        return False, "digital products cannot be returned once delivered"
    days = (date.today() - o["delivered_on"]).days
    if days > 30:
        return False, f"it was delivered {days} days ago, outside the 30-day window"
    return True, f"delivered {days} days ago, inside the 30-day window"


@tool(parse_docstring=True)
def lookup_order(order_id: str) -> str:
    """Look up an order's item, status, and delivery date.

    Args:
        order_id: The order ID, for example "A-1001".
    """
    o = get_order(order_id)
    delivered = o["delivered_on"].isoformat() if o["delivered_on"] else "not yet delivered"
    return f"{order_id.upper()}: {o['item']} | status: {o['status']} | delivered: {delivered}"


@tool(parse_docstring=True)
def check_refund_eligibility(order_id: str) -> str:
    """Check whether an order can be refunded under the 30-day return policy. Read-only.

    Args:
        order_id: The order ID, for example "A-1001".
    """
    ok, reason = refund_check(order_id)
    return f"{'Eligible' if ok else 'Not eligible'}: {reason}."


@tool(parse_docstring=True)
def issue_refund(order_id: str) -> str:
    """Issue a refund for an order. This moves money, so only call it after check_refund_eligibility says Eligible.

    Args:
        order_id: The order ID, for example "A-1001".
    """
    oid = order_id.strip().upper()
    ok, reason = refund_check(oid)               # re-check here: never trust that the model already did
    if not ok:
        raise ToolException(f"Refund refused: {reason}.")
    if oid in REFUNDED:
        return f"{oid} was already refunded."
    REFUNDED.append(oid)
    return f"Refund issued for {oid}."


@tool(parse_docstring=True)
def escalate_to_human(order_id: str, summary: str) -> str:
    """Hand a case to a human agent when the assistant cannot resolve it.

    Args:
        order_id: The order the case is about.
        summary: One or two sentences describing the problem and what was already tried.
    """
    ticket_id = f"T-{len(TICKETS) + 1:03d}"
    TICKETS.append({"ticket": ticket_id, "order_id": order_id, "summary": summary})
    return f"Escalated. Ticket {ticket_id} created; a human will follow up."


support_tools = [lookup_order, check_refund_eligibility, issue_refund, escalate_to_human]

SYSTEM_PROMPT = (
    "You are a support assistant for an online store. Use tools to look up facts; never guess about "
    "orders or refunds. Check refund eligibility before issuing any refund. If a refund is not possible, "
    "explain why in one or two sentences and offer to escalate to a human. Be brief and friendly."
)


def print_messages(result: dict) -> None:
    """Print an agent result's messages as a compact trace."""
    for m in result["messages"]:
        kind = type(m).__name__.replace("Message", "")
        flag = " [error]" if getattr(m, "status", None) == "error" else ""
        text = str(m.content).strip()
        calls = [f"{c['name']}({json.dumps(c['args'])})" for c in getattr(m, "tool_calls", [])]
        if text or not calls:
            print(f"  {kind:9}{flag} {text[:170]}")
        for c in calls:
            print(f"  {kind:9} -> calls {c}")

print("Toolbox ready:", [t.name for t in support_tools])


Toolbox ready: ['lookup_order', 'check_refund_eligibility', 'issue_refund', 'escalate_to_human']


---
## 2. Your first agent

All the loop code from `12` — `run_tool_loop`, `execute_tool_call`, the message bookkeeping — is replaced
by one function call. The result is a dict whose `"messages"` is the same list you built by hand.

In [4]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

agent = create_agent(llm, tools=support_tools, system_prompt=SYSTEM_PROMPT)

result = agent.invoke({"messages": [{"role": "user", "content": "Where is my order A-1002?"}]})
print_messages(result)


  Human     Where is my order A-1002?
  AI        -> calls lookup_order({"order_id": "A-1002"})
  Tool      A-1002: Running shoes | status: delivered | delivered: 2026-08-27
  AI        Your order A‑1002 (running shoes) was delivered on August 27, 2026. Let me know if you need anything else!


Look at what the agent *is*: a graph with two working nodes, wired as a loop. (This uses
`get_graph()` directly, so it needs no extra drawing packages.)

In [5]:
graph = agent.get_graph()
print("nodes:", list(graph.nodes))
for e in graph.edges:
    kind = "when needed" if e.conditional else "always"
    print(f"  {e.source:10} -> {e.target:10} ({kind})")


nodes: ['__start__', 'model', 'tools', '__end__']
  __start__  -> model      (always)
  model      -> __end__    (when needed)
  model      -> tools      (when needed)
  tools      -> model      (when needed)


That is the `12` loop, drawn: `model` decides; if it asked for tools, go to `tools` and come back to
`model`; if not, end. `create_agent` is a convenience for building exactly this graph.

> ⚠️ **If you've seen older tutorials:** `create_react_agent` (from `langgraph.prebuilt`) still imports
> but prints a deprecation warning telling you to use `create_agent`. `AgentExecutor` is no longer in
> `langchain.agents` at all — it lives in `langchain_classic.agents`. New code should use `create_agent`.

---
## 3. Your loop's protections, and what the agent gives you

Every protection in the `12` loop has a counterpart. What matters is which are **on by default**:

| Your `12` loop | In `create_agent` | By default? |
|---|---|---|
| Call → run tools → return results → repeat | `create_agent(model, tools)` and `agent.invoke(...)` | ✅ built in |
| Bad arguments / unknown tool → error message | Handled by the tools node | ✅ yes (Section 4) |
| An exception **inside a tool** → error message | `ToolErrorMiddleware`, or `handle_tool_error=True` on the tool | ❌ **no — the run crashes** (Section 4) |
| `max_steps` | `ModelCallLimitMiddleware`, `ToolCallLimitMiddleware`, or `config={"recursion_limit": N}` | ❌ **no — default limit is 10007** (Section 5) |
| `invoke_with_retry` | `ModelRetryMiddleware` | ❌ opt-in (Section 6) |
| `needs_approval` + `approve` | `HumanInTheLoopMiddleware` + a checkpointer | ❌ opt-in (Section 7) |
| Passing the same `messages` list to the next turn | `checkpointer=InMemorySaver()` plus a `thread_id` | ❌ opt-in (Section 8) |

**Middleware** is the agent's word for code that wraps a step — the model call or a tool call — to
observe it, limit it, retry it or gate it. Your `LangGraph/07_Middleware` notebooks write their own with
`@wrap_tool_call`; the built-in ones used below cover the common cases.

---
## 4. A scripted model, so failures happen on demand

To *see* these behaviours you need a model that misbehaves exactly when told to. This stand-in replays a
fixed list of replies. It has the same interface as a real chat model, so `create_agent` can't tell the
difference — which is also how you'd unit-test an agent without paying for API calls.

In [6]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatGeneration, ChatResult


class ScriptedModel(BaseChatModel):
    """Replays a list of AIMessages (the last one repeats). Returns a FRESH COPY on every call."""
    script: list
    position: int = 0

    @property
    def _llm_type(self) -> str:
        return "scripted"

    def bind_tools(self, tools, **kwargs):
        return self

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        # Why the copy: the agent gives every message an id. Handing back the SAME object twice makes the
        # second one look like an UPDATE to the first, so the history silently stops growing.
        reply = self.script[min(self.position, len(self.script) - 1)].model_copy(deep=True)
        reply.id = None
        for tc in reply.tool_calls:
            tc["id"] = f"{tc['id']}_{self.position}"         # unique call ids
        self.position += 1
        return ChatResult(generations=[ChatGeneration(message=reply)])


def make_call(name, args, cid):
    return {"name": name, "args": args, "id": cid, "type": "tool_call"}

def lab_agent(script, **kwargs):
    return create_agent(ScriptedModel(script=script), tools=support_tools, **kwargs)

def ask(agent, text, config=None):
    return agent.invoke({"messages": [{"role": "user", "content": text}]}, config=config)

print("Lab ready.")


Lab ready.


### 4a. When the *model* makes a mistake — handled for you

In [7]:
print("=== The model forgets a required argument ===")
print_messages(ask(lab_agent([
    AIMessage(content="", tool_calls=[make_call("lookup_order", {}, "c1")]),
    AIMessage(content="Sorry, which order number do you mean?"),
]), "Where is my order?"))

print("\n=== The model asks for a tool that does not exist ===")
print_messages(ask(lab_agent([
    AIMessage(content="", tool_calls=[make_call("delete_everything", {}, "c1")]),
    AIMessage(content="I can't do that, sorry."),
]), "Delete all the orders."))


=== The model forgets a required argument ===
  Human     Where is my order?
  AI        -> calls lookup_order({})
  Tool      [error] Error invoking tool 'lookup_order' with kwargs {} with error:
 order_id: Field required
 Please fix the error and try again.
  AI        Sorry, which order number do you mean?

=== The model asks for a tool that does not exist ===
  Human     Delete all the orders.
  AI        -> calls delete_everything({})
  Tool      [error] Error: delete_everything is not a valid tool, try one of [lookup_order, check_refund_eligibility, issue_refund, escalate_to_human].
  AI        I can't do that, sorry.


Both became an `[error]` message that the model read and recovered from — the agent did that on its
own, with no setup.

### 4b. When your *tool* raises — the run crashes

In [8]:
crashing = lab_agent([
    AIMessage(content="", tool_calls=[make_call("lookup_order", {"order_id": "Z-9999"}, "c1")]),
    AIMessage(content="I couldn't find that order."),
])

try:
    print_messages(ask(crashing, "Where is order Z-9999?"))
except ToolException as e:
    print("The WHOLE run crashed with ToolException:", e)


The WHOLE run crashed with ToolException: No order found with ID 'Z-9999'. Ask the customer to double-check the order number.


`lookup_order` raised `ToolException` for an order that doesn't exist — a perfectly normal event —
and the agent gave up instead of letting the model say "I couldn't find that". Two fixes:

1. **`ToolErrorMiddleware`**, a handler you write once for every tool. It receives the exception *and*
   the request, and returns text to hand the model, or `None` to let the exception propagate. Returning
   `None` for anything you didn't anticipate is deliberate: unexpected errors should stay loud. Return
   text that **names the exception type** rather than dumping raw internals; the docs warn that raw
   exception messages can leak sensitive detail.
2. **`handle_tool_error=True` on the tool itself**, from `11_Tools` — checked while building this
   notebook to work inside an agent as well.

In [9]:
def explain_tool_error(error: Exception, request) -> str | None:
    if isinstance(error, ToolException):                 # an EXPECTED failure: tell the model
        return f"{type(error).__name__}: {error}"
    return None                                          # anything unexpected still stops the run, visibly

forgiving = lab_agent([
    AIMessage(content="", tool_calls=[make_call("lookup_order", {"order_id": "Z-9999"}, "c1")]),
    AIMessage(content="I couldn't find that order. Could you double-check the number?"),
], middleware=[ToolErrorMiddleware(on_error=explain_tool_error)])

print_messages(ask(forgiving, "Where is order Z-9999?"))


  Human     Where is order Z-9999?
  AI        -> calls lookup_order({"order_id": "Z-9999"})
  Tool      [error] ToolException: No order found with ID 'Z-9999'. Ask the customer to double-check the order number.
  AI        I couldn't find that order. Could you double-check the number?


---
## 5. Limits — your `max_steps`

A confused model can ask for tools forever. Start with the number that surprises people: with no limit
set, LangGraph allows **10007** steps. A scripted "runaway" model shows the three ways to cap it.

In [10]:
from langgraph.errors import GraphRecursionError

runaway = [AIMessage(content="", tool_calls=[make_call("lookup_order", {"order_id": "A-1001"}, "c1")])]

# 1. The blunt tool: a recursion limit in the config. It RAISES when reached.
try:
    ask(lab_agent(runaway), "go", config={"recursion_limit": 6})
except GraphRecursionError as e:
    print("recursion_limit=6 ->", type(e).__name__, "-", str(e).splitlines()[0][:90])


recursion_limit=6 -> GraphRecursionError - Recursion limit of 6 reached without hitting a stop condition. You can increase the limit 


In [11]:
# 2. ModelCallLimitMiddleware: counts MODEL calls, and can end gracefully with a message...
print("=== exit_behavior='end': stops politely ===")
print_messages(ask(lab_agent(runaway, middleware=[ModelCallLimitMiddleware(run_limit=3, exit_behavior="end")]), "go"))

# ...or raise, if you would rather catch it in your own code.
print("\n=== exit_behavior='error': raises ===")
try:
    ask(lab_agent(runaway, middleware=[ModelCallLimitMiddleware(run_limit=3, exit_behavior="error")]), "go")
except Exception as e:
    print(type(e).__name__, "-", e)


=== exit_behavior='end': stops politely ===
  Human     go
  AI        -> calls lookup_order({"order_id": "A-1001"})
  Tool      A-1001: Wireless earbuds | status: delivered | delivered: 2026-10-01
  AI        -> calls lookup_order({"order_id": "A-1001"})
  Tool      A-1001: Wireless earbuds | status: delivered | delivered: 2026-10-01
  AI        -> calls lookup_order({"order_id": "A-1001"})
  Tool      A-1001: Wireless earbuds | status: delivered | delivered: 2026-10-01
  AI        Model call limits exceeded: run limit (3/3)

=== exit_behavior='error': raises ===
ModelCallLimitExceededError - Model call limits exceeded: run limit (3/3)


In [12]:
# 3. ToolCallLimitMiddleware: counts TOOL calls. Useful for expensive tools (a paid API, an email sender).
print("=== ToolCallLimitMiddleware(run_limit=2, exit_behavior='end') ===")
print_messages(ask(lab_agent(runaway, middleware=[ToolCallLimitMiddleware(run_limit=2, exit_behavior="end")]), "go"))


=== ToolCallLimitMiddleware(run_limit=2, exit_behavior='end') ===
  Human     go
  AI        -> calls lookup_order({"order_id": "A-1001"})
  Tool      A-1001: Wireless earbuds | status: delivered | delivered: 2026-10-01
  AI        -> calls lookup_order({"order_id": "A-1001"})
  Tool      A-1001: Wireless earbuds | status: delivered | delivered: 2026-10-01
  AI        -> calls lookup_order({"order_id": "A-1001"})
  Tool      [error] Tool call limit exceeded. Do not make additional tool calls.
  AI        Tool call limit reached: run limit exceeded (3/2 calls).


Use `"end"` for a user-facing assistant (the user gets a sentence, not a stack trace) and `"error"` in
batch jobs where you want the failure to be noticed.

---
## 6. Retries — your `invoke_with_retry`

Models fail transiently: a rate limit, a timeout, Groq's occasional `tool_use_failed`. `ModelRetryMiddleware`
retries with exponential backoff. (The tiny `initial_delay` below only keeps this demo quick; the default
is a full second.)

In [13]:
class FlakyModel(ScriptedModel):
    """Fails the first `fails` calls with a provider-style error, then behaves."""
    fails: int = 1

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        if self.fails > 0:
            self.fails -= 1
            raise RuntimeError("Error code: 400 - tool_use_failed: Failed to call a function.")
        return super()._generate(messages, stop, run_manager, **kwargs)


flaky = create_agent(
    FlakyModel(script=[AIMessage(content="Recovered after one transient failure.")]),
    tools=support_tools,
    middleware=[ModelRetryMiddleware(max_retries=2, initial_delay=0.01, jitter=False)],
)
print_messages(ask(flaky, "hello"))


  Human     hello
  AI        Recovered after one transient failure.


---
## 7. Approval gates — your `needs_approval`

`HumanInTheLoopMiddleware` pauses the agent *before* a named tool runs. Pausing means the run must be
**saved somewhere to resume from**, so it needs a `checkpointer`, and every call needs a `thread_id`
(which conversation this is). Tools set to `False` run freely.

When it pauses, the result contains an `__interrupt__` instead of a final answer. Resume by sending
`Command(resume={"decisions": [...]})`, with **exactly one decision per paused action** (a mismatch
raises `ValueError`).

In [14]:
refund_script = [
    AIMessage(content="", tool_calls=[make_call("issue_refund", {"order_id": "A-1001"}, "c1")]),
    AIMessage(content="All done — let me know if you need anything else."),
]

def refund_agent():
    return lab_agent(
        refund_script,
        middleware=[HumanInTheLoopMiddleware(interrupt_on={"issue_refund": True, "lookup_order": False})],
        checkpointer=InMemorySaver(),
    )

REFUNDED.clear()
approval_agent = refund_agent()
cfg = {"configurable": {"thread_id": "demo-approve"}}

paused = ask(approval_agent, "Please refund A-1001.", config=cfg)
print("result keys     :", list(paused.keys()))
request = paused["__interrupt__"][0].value
print("waiting on      :", [(a["name"], a["args"]) for a in request["action_requests"]])
print("allowed choices :", request["review_configs"][0]["allowed_decisions"])
print("refund executed?:", REFUNDED, "   <- nothing has run yet")
print("paused at node  :", approval_agent.get_state(cfg).next)


result keys     : ['messages', '__interrupt__']
waiting on      : [('issue_refund', {'order_id': 'A-1001'})]
allowed choices : ['approve', 'edit', 'reject', 'respond']
refund executed?: []    <- nothing has run yet
paused at node  : ('HumanInTheLoopMiddleware.after_model',)


In [15]:
# APPROVE: the tool now runs.
done = approval_agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=cfg)
print_messages(done)
print("refund executed?:", REFUNDED)


  Human     Please refund A-1001.
  AI        -> calls issue_refund({"order_id": "A-1001"})
  Tool      Refund issued for A-1001.
  AI        All done — let me know if you need anything else.
refund executed?: ['A-1001']


In [16]:
# REJECT: the tool never runs, and the model is told why.
REFUNDED.clear()
agent2 = refund_agent(); cfg2 = {"configurable": {"thread_id": "demo-reject"}}
ask(agent2, "Please refund A-1001.", config=cfg2)
done = agent2.invoke(Command(resume={"decisions": [{"type": "reject", "message": "Manager says no refunds today."}]}), config=cfg2)
print_messages(done)
print("refund executed?:", REFUNDED)

# EDIT: the reviewer changes the arguments, and THEIR version is what runs.
REFUNDED.clear()
agent3 = refund_agent(); cfg3 = {"configurable": {"thread_id": "demo-edit"}}
ask(agent3, "Please refund A-1001.", config=cfg3)
agent3.invoke(Command(resume={"decisions": [{"type": "edit",
              "edited_action": {"name": "issue_refund", "args": {"order_id": "A-1005"}}}]}), config=cfg3)
print("\nafter EDIT, refund executed for:", REFUNDED, "  <- the reviewer's order, not the model's")


  Human     Please refund A-1001.
  AI        -> calls issue_refund({"order_id": "A-1001"})
  Tool      [error] User rejected the tool call for `issue_refund` with reason: Manager says no refunds today.
  AI        All done — let me know if you need anything else.
refund executed?: []

after EDIT, refund executed for: ['A-1005']   <- the reviewer's order, not the model's


(The pause also lists a fourth choice, `respond`, which this notebook doesn't use or test.)

Two things to take from the edit case. It is a real power — and also a reason the **tool must still
enforce its own rules**: a reviewer (or whoever edits the payload) could send *any* order, and
`issue_refund` re-checks eligibility regardless.

Your `LangGraph/08_HITL` notebooks build the same idea at the graph level with `interrupt`; this
middleware is the ready-made version for tool calls.

---
## 8. Memory — your growing `messages` list

In `12` you kept the conversation alive by reusing the same `messages` list. An agent has no list for
you to hold; instead a **checkpointer** saves the state after each step, keyed by `thread_id`.

In [17]:
memory_agent = create_agent(
    ScriptedModel(script=[
        AIMessage(content="Nice to meet you, Asha."),
        AIMessage(content="Your name is Asha."),
        AIMessage(content="I don't know your name yet."),
    ]),
    tools=support_tools,
    checkpointer=InMemorySaver(),
)

alice = {"configurable": {"thread_id": "alice"}}
bob   = {"configurable": {"thread_id": "bob"}}

r1 = ask(memory_agent, "Hi, I'm Asha.", config=alice)
r2 = ask(memory_agent, "What's my name?", config=alice)      # same thread: history carries over
r3 = ask(memory_agent, "What's my name?", config=bob)        # new thread: starts empty

print("messages in alice's thread after turn 1:", len(r1["messages"]))
print("messages in alice's thread after turn 2:", len(r2["messages"]), "  <- grew: the thread remembers")
print("messages in bob's thread after turn 1  :", len(r3["messages"]), "  <- separate, started fresh")


messages in alice's thread after turn 1: 2
messages in alice's thread after turn 2: 4   <- grew: the thread remembers
messages in bob's thread after turn 1  : 2   <- separate, started fresh


`InMemorySaver` lives in the Python process, so it forgets when the process ends. For real
deployments you swap in a database-backed saver — that, and memory that outlives a single conversation,
are `LangGraph/09_Memory_Persistence` and `12_Long_Term_Memory`.

---
## Real-world capstone: the support agent, assembled

Everything above in one agent: retries for flaky model calls, a cap on runaway loops, expected tool
failures turned into messages, and a human gate on the one tool that moves money. Compare it with the
~50 lines of loop code in `12` — the safeguards are the same, but now they are *declared*.

In [18]:
REFUNDED.clear(); TICKETS.clear()

support_agent = create_agent(
    llm,
    tools=support_tools,
    system_prompt=SYSTEM_PROMPT,
    middleware=[
        ModelRetryMiddleware(max_retries=2),                              # flaky model calls
        ModelCallLimitMiddleware(run_limit=8, exit_behavior="end"),       # runaway loops
        ToolErrorMiddleware(on_error=explain_tool_error),                 # expected tool failures -> messages
        HumanInTheLoopMiddleware(interrupt_on={"issue_refund": True}),    # the only tool that moves money
    ],
    checkpointer=InMemorySaver(),
)


def pending_actions(agent, config) -> list[dict]:
    state = agent.get_state(config)
    return [req for task in state.tasks for intr in task.interrupts for req in intr.value["action_requests"]]


def report(agent, result, config):
    if "__interrupt__" in result:
        for req in pending_actions(agent, config):
            print(f"  [PAUSED] approval needed: {req['name']}({json.dumps(req['args'])})")
    else:
        print("  Assistant:", result["messages"][-1].content)


def chat(agent, text, thread_id):
    config = {"configurable": {"thread_id": thread_id}}
    print(f"You: {text}")
    report(agent, ask(agent, text, config=config), config)


def decide(agent, thread_id, decision):
    config = {"configurable": {"thread_id": thread_id}}
    n = len(pending_actions(agent, config))              # exactly one decision per paused action
    print(f"Human decides: {decision['type']}" + (f" ({decision['message']})" if "message" in decision else ""))
    report(agent, agent.invoke(Command(resume={"decisions": [decision] * n}), config=config), config)

print("Support agent assembled.")


Support agent assembled.


In [19]:
# --- Conversation 1: an eligible refund, approved, followed by a memory check on the same thread ---
chat(support_agent, "Hi! Can you refund order A-1001? The earbuds were a gift I don't need.", "alice")
decide(support_agent, "alice", {"type": "approve"})
print()
chat(support_agent, "Thanks! Which order did you just refund for me?", "alice")
print("\nRefunded so far:", REFUNDED)


You: Hi! Can you refund order A-1001? The earbuds were a gift I don't need.
  [PAUSED] approval needed: issue_refund({"order_id": "A-1001"})
Human decides: approve
  Assistant: Your refund for order **A-1001** has been processed. You should see the amount back in your original payment method within a few business days. If you need anything else, just let me know!

You: Thanks! Which order did you just refund for me?
  Assistant: I just processed a refund for **order A-1001**. Let me know if there's anything else I can help with!

Refunded so far: ['A-1001']


In [20]:
# --- Conversation 2: an eligible refund, but the human says no ---
chat(support_agent, "Please refund order A-1005, the laptop sleeve doesn't fit.", "bob")
decide(support_agent, "bob", {"type": "reject", "message": "Policy review pending for this account."})
print("\nRefunded so far:", REFUNDED, "  <- A-1005 must NOT be here")

# --- Conversation 3: an order that doesn't exist (the tool raises; the middleware turns it into a message) ---
print()
chat(support_agent, "Where is my order Z-9999?", "carol")


You: Please refund order A-1005, the laptop sleeve doesn't fit.
  [PAUSED] approval needed: issue_refund({"order_id": "A-1005"})
Human decides: reject (Policy review pending for this account.)
  Assistant: I’m sorry, but I can’t process the refund automatically right now. I’ll forward your request to a human agent who can complete the refund for you. Thank you for your patience!

Refunded so far: ['A-1001']   <- A-1005 must NOT be here

You: Where is my order Z-9999?
  Assistant: I’m sorry—I couldn’t find an order with the ID **Z‑9999**. Could you please double‑check the order number and let me know?


---
## When to use what

| Use | When |
|---|---|
| **A hand-written loop** (`12`) | Learning, tiny scripts, or when you need control `create_agent` doesn't expose. You own every line, including the bugs |
| **`create_agent`** (this notebook) | The default for a tool-using assistant: loop, state, limits, retries and approvals are declared, not written |
| **LangGraph graphs** (your `LangGraph/` folders) | Distinct steps with branching, parallel work, subgraphs, multiple agents, or custom state — beyond one model↔tools loop |

**Other pieces in the same package** (present in `langchain 1.4.4`; not exercised here — see the docs and
your `LangGraph/07_Middleware`): `SummarizationMiddleware` (long conversations), `ModelFallbackMiddleware`,
`PIIMiddleware`, `TodoListMiddleware`, `ContextEditingMiddleware`. `create_agent` also takes, per its
signature, `response_format` (a structured final answer, as in `05_Structured_Output`), `store`
(long-term memory), and `context_schema`.

### Try it yourself
1. Delete `ToolErrorMiddleware` from `support_agent` and rerun conversation 3. What does the customer
   experience now, compared with what the middleware gave them?
2. Set `ModelCallLimitMiddleware(run_limit=2, ...)` and ask a question that needs three tool calls.
3. Add `check_refund_eligibility` to `interrupt_on` too. Does the conversation still work end to end?
4. Make `decide` approve *every* paused action. Why is that a bad default in a real system?
5. Write your own middleware with `@wrap_tool_call` (look at `LangGraph/07_Middleware`) that logs every
   tool call's name and arguments.
6. Swap `ChatGroq` for `ChatOpenAI`. Does anything else in the notebook need to change?

---
**Next:** your `LangGraph/` folders pick up exactly here — `05_Tool_Calling`, `07_Middleware`, `08_HITL`
and `09_Memory_Persistence` are the graph-level versions of what this notebook declared in a line each.
Back in `LangChain/`, `14_Error_Handling` covers the *non-agent* equivalents for ordinary chains:
retrying and falling back around any runnable.
